# Basic Web-Scraping: Direct from HTML

This workbook showcases a very basic scraping project, extracting data from a HTML file `home.html` which mimics a simple website.

The "website" in question represents a highly simplified version of a educational platform which offers courses on Python programming. The purpose of scraping the site is to extract the course names and their associated prices.

The `BeautifulSoup` and `lxml` libraries are used to parse the HTML file to extract the relevant data.

## Importing Libraries

In [1]:
from bs4 import BeautifulSoup
import lxml
import pandas as pd

## Reading the File Contents

In [2]:
with open("home.html", "r") as html_file:
  content = html_file.read() # read the content of the HTML file
  soup = BeautifulSoup(content, "lxml") # create an instance of BeautifulSoup and parse the HTML content using the lxml parser
  print(soup.prettify()) # print the prettified version of the parsed HTML content, which formats the HTML with proper indentation and line breaks for better readability

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>
  <link crossorigin="anonymous" href="https://stackpath.bootstrapcdn.com/bootstrap/4.5.2/css/bootstrap.min.css" integrity="sha384-JcKb8q3iqJ61gNV9KGb8thSsNjpSL0n8PARn9HuZOnIxN0hoP+VmmDGMN5t9UJ0Z" rel="stylesheet"/>
  <title>
   My Courses
  </title>
 </head>
 <body>
  <h1>
   Hello, Start Learning!
  </h1>
  <div class="card" id="card-python-for-beginners">
   <div class="card-header">
    Python
   </div>
   <div class="card-body">
    <h5 class="card-title">
     Python for beginners
    </h5>
    <p class="card-text">
     If you are new to Python, this is the course that you should buy!
    </p>
    <a class="btn btn-primary" href="#">
     Start for 20$
    </a>
   </div>
  </div>
  <div class="card" id="card-python-web-development">
   <div class="card-header">
    Python
   </div>
   <div class="card-body">
    <h5 class="ca

The data of interest is stored within the **HTML elements**. In this case (and most cases) the data is nested in div tags (which are used to define *divisions* or *sections* in a HTML document).

The `class` attributes define the CSS styling that should be applied to each HTML element (in this example, CSS styling is applied externally through a link tag in the head tag of the file). This is not relevant for data retrieval, but is good to be aware of.

## BeautifulSoup Methods

The `.find()` method returns the first instance of the tag passed to the method.

In [3]:
first_h5_tag = soup.find("h5")
print(first_h5_tag, type(first_h5_tag)) # print the first h5 tag and its type

<h5 class="card-title">Python for beginners</h5> <class 'bs4.element.Tag'>


To return all tags of a particular type, `.find_all()` must be used.

In [4]:
h5_tags = soup.find_all("h5")
h5_tags

[<h5 class="card-title">Python for beginners</h5>,
 <h5 class="card-title">Python Web Development</h5>,
 <h5 class="card-title">Python Machine Learning</h5>]

Each tag has a corresponding `text attribute` which can be used to extract text from each tag:

In [5]:
courses = h5_tags

for course in courses:
  print(course.text) # print the text content of each h5 tag, which represents the course names

Python for beginners
Python Web Development
Python Machine Learning


The goal of scraping this file is to retrieve all courses and their associated prices. If we return to the HTML code, we can see that all of this information is contained in `<div class="card">` tags. We can therefore write code to find all of the div tags with a class attribute of "card". *Note,* an underscore is required when specifiying the class argument to filter by (since class is a Python keyword).

In [6]:
course_cards = soup.find_all("div", class_="card")
course_cards[0] # print the first course card, which is a div element with the class "card"

<div class="card" id="card-python-for-beginners">
<div class="card-header">
            Python
         </div>
<div class="card-body">
<h5 class="card-title">Python for beginners</h5>
<p class="card-text">If you are new to Python, this is the course that you should buy!</p>
<a class="btn btn-primary" href="#">Start for 20$</a>
</div>
</div>

Each course card in the list of course_cards contains the information we require. We can now iterate over the course_cards and extract the relevant data.

In [7]:
course_names = []
course_prices = []

for course_card in course_cards:
  course_name = course_card.h5.text # extract the text content of the h5 tag within each course card, which represents the course name
  course_price = course_card.a.text.split()[-1]
  
  course_names.append(course_name) 
  course_prices.append(course_price) 
  
courses = dict(zip(course_names, course_prices))
print(courses) 

{'Python for beginners': '20$', 'Python Web Development': '50$', 'Python Machine Learning': '100$'}


In [8]:
df = pd.DataFrame(courses.items(), columns=["course", "price_usd"])
df

,course,price_usd
0,Python for beginners,20$
1,Python Web Development,50$
2,Python Machine Learning,100$


In [9]:
df["price_usd"] = df["price_usd"].str.replace("$", "").astype(float)

In [10]:
df

,course,price_usd
0,Python for beginners,20.0
1,Python Web Development,50.0
2,Python Machine Learning,100.0
